In [1]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langchain.tools import tool
from functools import partial

from typing_extensions import Annotated, TypedDict
from dotenv import load_dotenv
from typing import Sequence
import os

load_dotenv()

os.environ["LANGCHAIN_TRACING"] = "false"
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_TRACING_V2"] = os.environ.get("LANGCHAIN_TRACING_V2")
os.environ["LANGCHAIN_API_KEY"] = os.environ.get("LANGCHAIN_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.environ.get("GOOGLE_API_KEY")


model = init_chat_model("gemini-2.5-flash-lite", model_provider="google_genai")

In [15]:

def simul_credit(revenus:int, montant:int, duree:int):
    """Trouve si un crédit est réalisable en fonction des revenus, du montant et de la durée.
    revenus: revenus mensuels en euros
    montant: montant du crédit en euros
    duree: durée du crédit en années
    Retourne un message indiquant si le crédit est réalisable ou non.
    """
    taux = 0.035  # Taux d'intérêt fixe de 3.5%
    mensualite = (montant * taux / 12) / (1 - (1 + taux / 12) ** (-duree * 12))
    quot_cessible = revenus * 0.42
    if mensualite > quot_cessible:
        return (f"Le montant de la mensualité : {mensualite}, dépasse la quotité cessible : {quot_cessible}. Crédit non réalisable.")
    else:
        return ("Crédit réalisable avec une mensualité de {:.2f} €".format(mensualite))
    
simul_credit(2300, 100000, 25)

'Crédit réalisable avec une mensualité de 500.62 €'

In [3]:
# 1. Simplified State & Chain
class State(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]
    language: str

# Use the pipe operator (|) to handle parsing automatically
# Note: Added {language} to the prompt so the variable is actually used
prompt = ChatPromptTemplate.from_messages([
    ("system", "Tu es un conseiller financier expert (High-Net-Worth). Réponds en {language}."
    " Utilise l'outil simul_credit pour aider à évaluer les demandes de crédit. Demande toujours "
    "son revenus mensuels, le montant du crédit et la durée en années."),
    MessagesPlaceholder(variable_name="messages"),
])

tools = [simul_credit]
model_with_tools = model.bind_tools(tools)

chain = prompt | model_with_tools

# 2. Optimized Node
async def call_model(state: State):
    # ainvoke returns the parsed string directly due to the pipe chain
    response = await chain.ainvoke(state)
    return {"messages": response}

# 3. Clean Graph Definition

workflow = StateGraph(State)
workflow.add_node("model", call_model)
workflow.add_edge(START, "model")
workflow.add_edge("model", END)

app = workflow.compile(checkpointer=MemorySaver())

# 4. Stream execution
config = {"configurable": {"thread_id": "abc1234"}}
inputs = {"messages": [HumanMessage("Bonsoir")], "language": "Français"}

## Corrected Streaming Loop
print("🤖 **Response:**", end=" ")

# Use stream_mode="values" to get the content delta instead of the full message list
async for event in app.astream(inputs, config, stream_mode="values"):
    if "messages" in event:
        message = event["messages"][-1]
        if isinstance(message, AIMessage) and message.content:
            print(message.content, end="", flush=True)

🤖 **Response:** Bonsoir ! En quoi puis-je vous aider ce soir ?

In [20]:
print("""k""")

k


In [19]:
config = {"configurable": {"thread_id": "abc1234"}}
inputs = {"messages": [HumanMessage("Alors ?")], "language": "Français"}

## Corrected Streaming Loop
print("🤖 **Response:**", end=" ")

# Use stream_mode="values" to get the content delta instead of the full message list
async for event in app.astream(inputs, config, stream_mode="values"):
    if "messages" in event:
        message = event["messages"][-1]
        if isinstance(message, AIMessage) and message.content:
            print(message.content, end="", flush=True)

🤖 **Response:** 

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash-lite' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite\nPlease retry in 2.803392655s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerPr